# Olist: giám sát drift cho model review xấu

Model review xấu được train trên đơn mua trước 2018-01-01 và chấm lúc đơn vừa giao xong. Giả sử nó
được đưa vào production từ tháng 1/2018. Notebook này đóng vai **job monitoring**: so dữ liệu mỗi
tháng với dữ liệu train (reference), tìm feature nào trôi, trôi bao nhiêu, và quan trọng nhất là
trôi đó có đáng lo không.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` đọc dữ liệu thật và gọi hàm; test bỏ qua chúng.
- Chấm: `uv run pytest tracks/data-science/modules/14-mlops-for-ds/exercises/intermediate/03-olist-drift-monitor -q`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
# Cell viết sẵn: giống bài Intermediate 01: bảng đơn giao xong có review, feature có sẵn lúc đơn vừa giao.
DATES = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
         "order_delivered_customer_date", "order_estimated_delivery_date"]
NUMERIC = ["promised_days", "delivery_days", "delay_days", "approve_hours", "n_items", "n_sellers",
           "price", "freight", "freight_ratio", "payment_installments"]
CATEGORICAL = ["customer_state", "payment_type"]
FEATURES = NUMERIC + CATEGORICAL
TARGET = "bad_review"


def load_olist_table(raw) -> pd.DataFrame:
    raw = Path(raw)
    orders = pd.read_csv(raw / "olist_orders_dataset.csv", parse_dates=DATES)
    reviews = pd.read_csv(raw / "olist_order_reviews_dataset.csv", parse_dates=["review_answer_timestamp"])
    customers = pd.read_csv(raw / "olist_customers_dataset.csv")
    items = pd.read_csv(raw / "olist_order_items_dataset.csv")
    payments = pd.read_csv(raw / "olist_order_payments_dataset.csv")

    delivered = orders[(orders.order_status == "delivered") & orders.order_delivered_customer_date.notna()]
    latest = reviews.sort_values("review_answer_timestamp", kind="stable").drop_duplicates("order_id", keep="last")
    df = delivered.merge(latest[["order_id", "review_score"]], on="order_id").merge(
        customers[["customer_id", "customer_state"]], on="customer_id")
    df[TARGET] = (df.review_score <= 2).astype(int)

    day = pd.Timedelta(days=1)
    df["promised_days"] = (df.order_estimated_delivery_date - df.order_purchase_timestamp) / day
    df["delivery_days"] = (df.order_delivered_customer_date - df.order_purchase_timestamp) / day
    df["delay_days"] = (df.order_delivered_customer_date - df.order_estimated_delivery_date) / day
    df["approve_hours"] = (df.order_approved_at - df.order_purchase_timestamp) / pd.Timedelta(hours=1)
    per_order = items.groupby("order_id").agg(
        n_items=("order_item_id", "size"), n_sellers=("seller_id", "nunique"),
        price=("price", "sum"), freight=("freight_value", "sum"))
    df = df.merge(per_order, on="order_id", how="left")
    df["freight_ratio"] = df.freight / (df.price + df.freight)
    main = payments.sort_values("payment_value", ascending=False, kind="stable").drop_duplicates("order_id")
    df = df.merge(main[["order_id", "payment_type", "payment_installments"]], on="order_id", how="left")
    df[CATEGORICAL] = df[CATEGORICAL].fillna("missing")
    df[NUMERIC] = df[NUMERIC].astype("float64")   # hợp đồng đầu vào: mọi cột số là float64
    return df.sort_values("order_purchase_timestamp", kind="stable").reset_index(drop=True)


def time_split(df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    t = df.order_purchase_timestamp
    return {"train": df[t < "2018-01-01"],
            "val": df[(t >= "2018-01-01") & (t < "2018-04-01")],
            "test": df[(t >= "2018-04-01") & (t < "2018-09-01")]}


def build_pipeline(C: float = 1.0) -> Pipeline:
    pre = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)])
    return make_pipeline(pre, LogisticRegression(C=C, max_iter=2000))

In [ ]:
# explore: đọc Olist thật (tải bằng: uv run python data/download_olist.py)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
table = load_olist_table(ROOT / "data" / "raw")
splits = time_split(table)
reference = splits["train"]
production = table[(table.order_purchase_timestamp >= "2018-01-01") & (table.order_purchase_timestamp < "2018-09-01")]
print(len(reference), len(production))

## 1. PSI cho biến số và biến phân loại

Chép `psi` và `psi_categorical` từ bài Basic 01 (cùng quy ước: bin theo quantile của reference,
biên ngoài là vô cực, chặn tỉ lệ bằng `eps = 1e-4`, bỏ NaN ở biến số, NaN là một giá trị riêng ở
biến phân loại).

In [ ]:
def psi(expected, actual, bins: int = 10, eps: float = 1e-4) -> float:
    """PSI của actual so với expected, đúng quy ước bài Basic 01."""
    # TODO
    raise NotImplementedError


def psi_categorical(expected, actual, eps: float = 1e-4) -> float:
    """PSI cho biến phân loại, NaN là một giá trị riêng."""
    # TODO
    raise NotImplementedError

## 2. Báo cáo drift cho một giai đoạn

`drift_report(reference, current, numeric, categorical) -> DataFrame`:

- index là tên feature (cả số lẫn phân loại), cột `psi`, `ks`, `level`;
- `ks` là `scipy.stats.ks_2samp(...).statistic` trên dữ liệu đã bỏ NaN, chỉ cho biến số (biến phân
  loại để NaN);
- `level` là `"stable"` (< 0.1), `"moderate"` (0.1 tới dưới 0.25), `"major"` (từ 0.25);
- sắp xếp theo `psi` giảm dần.

In [ ]:
def drift_report(reference: pd.DataFrame, current: pd.DataFrame, numeric: list[str],
                 categorical: list[str]) -> pd.DataFrame:
    """Bảng drift theo feature: psi, ks (chỉ biến số), level; sắp theo psi giảm dần."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: train (reference) so với toàn bộ giai đoạn test 2018-04 tới 2018-08
drift_report(reference, splits["test"], NUMERIC, CATEGORICAL).round(4)

In [ ]:
# explore: freight trôi mạnh nhất. Nhìn phân phối trước khi kết luận.
fig, ax = plt.subplots(figsize=(7, 3))
bins = np.linspace(0, 60, 61)
ax.hist(reference.freight, bins=bins, density=True, alpha=0.5, label="train (trước 2018)")
ax.hist(splits["test"].freight, bins=bins, density=True, alpha=0.5, label="test (2018-04 → 08)")
ax.set_xlabel("freight (BRL / đơn)")
ax.legend()
plt.show()
print(reference.freight.median(), splits["test"].freight.median())

## 3. Theo dõi theo tháng và cảnh báo

Job monitoring chạy định kỳ, nên ta cần chuỗi theo thời gian:

- `monthly_psi(reference, current, features, date_col="order_purchase_timestamp") -> DataFrame`:
  index là tháng dạng chuỗi `"YYYY-MM"` (theo `date_col`) theo thứ tự thời gian, một cột PSI cho mỗi
  feature số trong `features`, cộng cột `n` (số đơn trong tháng).
- `alerts(monthly, threshold=0.25) -> list[tuple[str, str]]`: các cặp `(tháng, feature)` có PSI từ
  `threshold` trở lên, sắp theo tháng rồi theo tên feature. Cột `n` không phải feature.

In [ ]:
def monthly_psi(reference: pd.DataFrame, current: pd.DataFrame, features: list[str],
                date_col: str = "order_purchase_timestamp") -> pd.DataFrame:
    """PSI từng tháng của current so với reference, cho từng feature số, kèm cột n."""
    # TODO: nhóm current theo tháng (dt.strftime("%Y-%m") hoặc dt.to_period("M"))
    raise NotImplementedError


def alerts(monthly: pd.DataFrame, threshold: float = 0.25) -> list[tuple[str, str]]:
    """Các cặp (tháng, feature) có PSI >= threshold."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: PSI theo tháng của các feature giao hàng, và các cảnh báo
watch = ["delay_days", "delivery_days", "promised_days", "freight", "price"]
monthly = monthly_psi(reference, production, watch)
print(monthly.round(3))
print(alerts(monthly))

In [ ]:
# explore: drift của chính score (prediction drift) và AUC theo tháng khi nhãn về
model = build_pipeline(1.0).fit(reference[FEATURES], reference[TARGET])
ref_score = model.predict_proba(reference[FEATURES])[:, 1]
prod = production.assign(score=model.predict_proba(production[FEATURES])[:, 1],
                         month=production.order_purchase_timestamp.dt.strftime("%Y-%m"),
                         late=production.delay_days > 0)
summary = prod.groupby("month").apply(lambda g: pd.Series({
    "late_rate": g.late.mean(), "bad_rate": g[TARGET].mean(), "mean_score": g.score.mean(),
    "psi_score": psi(ref_score, g.score), "auc": roc_auc_score(g[TARGET], g.score),
    "auc_on_time": roc_auc_score(g[TARGET][~g.late], g.score[~g.late]),
}), include_groups=False)
summary.round(3)

In [ ]:
# explore: AUC toàn cục so với AUC trong từng nhóm, theo giai đoạn
for name in ["train", "val", "test"]:
    g = splits[name]
    s = model.predict_proba(g[FEATURES])[:, 1]
    late = (g.delay_days > 0).to_numpy()
    print(f"{name:5s} trễ {late.mean():.3f} | AUC {roc_auc_score(g[TARGET], s):.3f} | "
          f"đúng hẹn {roc_auc_score(g[TARGET][~late], s[~late]):.3f} | trễ {roc_auc_score(g[TARGET][late], s[late]):.3f}")

## 4. Viết khuyến nghị monitoring

Viết 6–10 câu cho đội vận hành model (không chấm tự động; so với lời giải mẫu):

1. Feature nào trôi "major" trong giai đoạn test, và đó là drift có hại, vô hại hay có lợi?
   (Gợi ý: tháng 6/2018 gần như không đơn nào trễ.)
2. AUC tháng 6 tới 8 thấp hơn tháng 2 và 3. Model tệ đi, hay tổng thể đổi? Dựa vào bảng AUC trong
   từng nhóm.
3. Bạn sẽ cảnh báo trên chỉ số nào, ngưỡng nào, ai nhận cảnh báo, và khi nào thì retrain?
4. Nhãn (review) về sau khi giao khoảng 2 ngày (trung vị). Điều đó ảnh hưởng gì tới việc theo dõi
   AUC?